# 02 — New York City: Data Exploration

**Milestone 1** — pulling a live sample from NYC Open Data to verify the real schema before we design the cross-city pipeline (Milestone 2).

- **Dataset:** NYPD Complaint Data (Current, Year-To-Date), NYC Open Data (Socrata/SODA API)
- **Dataset ID:** `5uac-w243`
- **Scope of this notebook:** a recent-window sample only. NYC also publishes a separate multi-decade "Historic" complaint dataset (`qgea-i56i`) with the same shared schema — that gets pulled properly in the Milestone 2 pipeline for the full time range.

See `docs/DATA_SOURCES.md` for background and **especially** `docs/ETHICS.md` — this dataset includes per-complaint suspect/victim demographic fields, and there is an explicit, negotiated policy on how (and how not) those may be used in this project.

In [1]:
import requests
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

In [2]:
DATASET_ENDPOINT = "https://data.cityofnewyork.us/resource/5uac-w243.json"
SAMPLE_SIZE = 20_000
PAGE_SIZE = 5_000

RAW_DATA_DIR = Path("../data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def fetch_socrata_sample(endpoint: str, total: int, page_size: int, order: str = ":id") -> pd.DataFrame:
    """Pull `total` rows from a Socrata endpoint via offset pagination, most-recent-first."""
    frames = []
    fetched = 0
    while fetched < total:
        limit = min(page_size, total - fetched)
        params = {
            "$limit": limit,
            "$offset": fetched,
            "$order": f"{order} DESC",
        }
        response = requests.get(endpoint, params=params, timeout=30)
        response.raise_for_status()
        page = response.json()
        if not page:
            break
        frames.append(pd.DataFrame(page))
        fetched += len(page)
        if len(page) < limit:
            break
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


nyc_df = fetch_socrata_sample(DATASET_ENDPOINT, SAMPLE_SIZE, PAGE_SIZE, order="cmplnt_fr_dt")
nyc_df.shape

(20000, 36)

In [4]:
nyc_df.dtypes

cmplnt_num           object
addr_pct_cd          object
boro_nm              object
cmplnt_fr_dt         object
cmplnt_fr_tm         object
cmplnt_to_dt         object
cmplnt_to_tm         object
crm_atpt_cptd_cd     object
hadevelopt           object
jurisdiction_code    object
juris_desc           object
ky_cd                object
law_cat_cd           object
loc_of_occur_desc    object
ofns_desc            object
parks_nm             object
patrol_boro          object
pd_cd                object
pd_desc              object
prem_typ_desc        object
rpt_dt               object
station_name         object
susp_age_group       object
susp_race            object
susp_sex             object
vic_age_group        object
vic_race             object
vic_sex              object
x_coord_cd           object
y_coord_cd           object
latitude             object
longitude            object
lat_lon              object
geocoded_column      object
transit_district     object
housing_psa         

In [5]:
nyc_df.head(3)

,cmplnt_num,addr_pct_cd,boro_nm,cmplnt_fr_dt,cmplnt_fr_tm,cmplnt_to_dt,cmplnt_to_tm,crm_atpt_cptd_cd,hadevelopt,jurisdiction_code,juris_desc,ky_cd,law_cat_cd,loc_of_occur_desc,ofns_desc,parks_nm,patrol_boro,pd_cd,pd_desc,prem_typ_desc,rpt_dt,station_name,susp_age_group,susp_race,susp_sex,vic_age_group,vic_race,vic_sex,x_coord_cd,y_coord_cd,latitude,longitude,lat_lon,geocoded_column,transit_district,housing_psa
0,322817026,47,BRONX,2026-03-31T00:00:00.000,16:20:00,2026-03-31T00:00:00.000,16:25:00,COMPLETED,(null),0,N.Y. POLICE DEPT,348,MISDEMEANOR,FRONT OF,VEHICLE AND TRAFFIC LAWS,(null),PATROL BORO BRONX,916,LEAVING SCENE-ACCIDENT-PERSONA,STREET,2026-03-31T00:00:00.000,(null),UNKNOWN,UNKNOWN,U,UNKNOWN,WHITE HISPANIC,E,1023961,265468,40.895243,-73.856362,"{'latitude': '40.895243', 'longitude': '-73.85...","{'type': 'Point', 'coordinates': [-73.856362, ...",NaN,NaN
1,322677136,47,BRONX,2026-03-31T00:00:00.000,11:40:00,2026-03-31T00:00:00.000,11:56:00,COMPLETED,(null),0,N.Y. POLICE DEPT,344,MISDEMEANOR,INSIDE,ASSAULT 3 & RELATED OFFENSES,(null),PATROL BORO BRONX,101,ASSAULT 3,RESIDENCE - APT. HOUSE,2026-03-31T00:00:00.000,(null),45-64,BLACK,M,25-44,BLACK,F,1027358,261997,40.8857,-73.844098,"{'latitude': '40.8857', 'longitude': '-73.8440...","{'type': 'Point', 'coordinates': [-73.844098, ...",NaN,NaN
2,322705208,52,BRONX,2026-03-31T00:00:00.000,18:36:00,NaN,(null),COMPLETED,(null),0,N.Y. POLICE DEPT,126,FELONY,OPPOSITE OF,MISCELLANEOUS PENAL LAW,(null),PATROL BORO BRONX,198,CRIMINAL CONTEMPT 1,STREET,2026-03-31T00:00:00.000,(null),45-64,BLACK HISPANIC,M,25-44,WHITE HISPANIC,F,1015719,257410,40.87316,-73.886211,"{'latitude': '40.87316', 'longitude': '-73.886...","{'type': 'Point', 'coordinates': [-73.886211, ...",NaN,NaN


## Data quality checks

In [6]:
null_rates = (nyc_df.isna().mean() * 100).round(2).sort_values(ascending=False)
null_rates[null_rates > 0]

transit_district    93.22
housing_psa         92.94
cmplnt_to_dt         3.79
pd_cd                0.04
dtype: float64

In [7]:
nyc_df["cmplnt_fr_dt"] = pd.to_datetime(nyc_df["cmplnt_fr_dt"], errors="coerce")
print("date range in this sample:", nyc_df["cmplnt_fr_dt"].min(), "to", nyc_df["cmplnt_fr_dt"].max())

date range in this sample: 2026-03-17 00:00:00 to 2026-03-31 00:00:00


In [8]:
nyc_df["ofns_desc"].value_counts().head(20)

ofns_desc
PETIT LARCENY                      3234
HARRASSMENT 2                      3083
ASSAULT 3 & RELATED OFFENSES       2393
CRIMINAL MISCHIEF & RELATED OF     1234
GRAND LARCENY                      1201
VEHICLE AND TRAFFIC LAWS           1149
FELONY ASSAULT                     1087
DANGEROUS DRUGS                     757
MISCELLANEOUS PENAL LAW             744
OTHER OFFENSES RELATED TO THEFT     686
OFF. AGNST PUB ORD SENSBLTY &       570
ROBBERY                             490
GRAND LARCENY OF MOTOR VEHICLE      432
BURGLARY                            381
OFFENSES AGAINST PUBLIC ADMINI      360
SEX CRIMES                          333
DANGEROUS WEAPONS                   331
FORGERY                             215
OTHER STATE LAWS                    201
CRIMINAL TRESPASS                   186
Name: count, dtype: int64

In [9]:
# law_cat_cd is NYC's severity classification (FELONY / MISDEMEANOR / VIOLATION) -- a useful confounder to control
# for later if we ever compare demographic fields, per docs/ETHICS.md.
nyc_df["law_cat_cd"].value_counts(dropna=False)

law_cat_cd
MISDEMEANOR    10762
FELONY          5991
VIOLATION       3247
Name: count, dtype: int64

In [10]:
# boro_nm gives us only 5 boroughs -- too coarse for neighborhood-level mapping. addr_pct_cd (precinct) is finer-grained
# but still not a named neighborhood; Milestone 2 will need a lat/lon -> Neighborhood Tabulation Area (NTA) spatial join
# using NYC's published NTA boundaries, the same technique as Chicago's community_area join.
print("unique boroughs:", nyc_df["boro_nm"].nunique(), nyc_df["boro_nm"].unique())
print("unique precincts:", nyc_df["addr_pct_cd"].nunique())
print("rows missing precinct:", nyc_df["addr_pct_cd"].isna().sum())

unique boroughs: 6 ['BRONX' 'MANHATTAN' 'QUEENS' 'BROOKLYN' '(null)' 'STATEN ISLAND']
unique precincts: 78
rows missing precinct: 0


## Demographic fields — handle with care (see `docs/ETHICS.md`)

We're only checking *availability* here (null rates, category values) — no comparisons or conclusions are drawn in this exploratory notebook. Any actual demographic comparison work happens later, under the explicit policy in `docs/ETHICS.md`: rates with population denominators, confounders controlled, no causal framing, presented as an optional reader-driven filter rather than the site's default narrative.

In [11]:
demo_cols = ["susp_race", "susp_sex", "susp_age_group", "vic_race", "vic_sex", "vic_age_group"]
for col in demo_cols:
    print(f"--- {col} ---")
    print(nyc_df[col].value_counts(dropna=False).head(8))
    print()

--- susp_race ---
susp_race
BLACK                             6760
UNKNOWN                           4949
WHITE HISPANIC                    3094
(null)                            1665
WHITE                             1536
BLACK HISPANIC                    1209
ASIAN / PACIFIC ISLANDER           748
AMERICAN INDIAN/ALASKAN NATIVE      39
Name: count, dtype: int64

--- susp_sex ---
susp_sex
M         11252
U          3835
F          3248
(null)     1665
Name: count, dtype: int64

--- susp_age_group ---
susp_age_group
UNKNOWN    6928
25-44      6689
45-64      2471
(null)     1665
18-24      1485
<18         415
65+         344
2026          1
Name: count, dtype: int64

--- vic_race ---
vic_race
UNKNOWN                           7597
BLACK                             4646
WHITE HISPANIC                    3147
WHITE                             2245
ASIAN / PACIFIC ISLANDER          1380
BLACK HISPANIC                     890
AMERICAN INDIAN/ALASKAN NATIVE      87
(null)                  

In [12]:
sample_path = RAW_DATA_DIR / "nyc_sample.csv"
nyc_df.to_csv(sample_path, index=False)
print(f"saved {len(nyc_df):,} rows to {sample_path}")

saved 20,000 rows to ../data/raw/nyc_sample.csv


## Findings

- **Sample:** 20,000 rows, most-recent-first, spans 2026-03-17 to 2026-03-31 — about two weeks. NYC's complaint volume is comparable in order of magnitude to Chicago's; the full historical pull belongs in a Milestone 2 pipeline job, not a notebook.

- **Null rates:** `transit_district` (93.2%) and `housing_psa` (92.9%) are effectively unusable columns for our purposes and should be dropped in the pipeline. `cmplnt_to_dt` (3.8%) is minor. Everything else is well-populated — but see the demographic-field note below, since NYC encodes "missing" as literal strings (`"(null)"`, `"UNKNOWN"`) rather than true nulls in several fields, so `isna()` alone understates how much is actually missing there.

- **Top offense categories (`ofns_desc`):** PETIT LARCENY (3,234), HARRASSMENT 2 (3,083), ASSAULT 3 & RELATED OFFENSES (2,393) lead. Severity split (`law_cat_cd`): 53.8% MISDEMEANOR, 30.0% FELONY, 16.2% VIOLATION — a genuinely useful confounder to control for if we ever do demographic comparisons, since severity plausibly correlates with both category and response handling.

- **Geography is coarser than Chicago's:** only 6 distinct `boro_nm` values (5 boroughs + a literal `"(null)"` string that needs cleaning in the pipeline), which is too coarse for neighborhood-level mapping on its own. `addr_pct_cd` (precinct) has 78 unique values with zero missing — a much better geography key, but still not a named neighborhood. Milestone 2 will need a lat/lon → Neighborhood Tabulation Area (NTA) spatial join using NYC's published boundary files, the same general technique as Chicago's `community_area` join, just via coordinates instead of a pre-assigned code.

- **Demographic fields exist but are heavily unknown/missing — this matters a lot for `docs/ETHICS.md`:** combining `"UNKNOWN"` and literal `"(null)"` strings, `susp_race` is unusable for **33.1%** of rows (6,614 / 20,000) and `vic_race` for **38.0%** (7,605 / 20,000). That's over a third of the data with no usable race field at all — a serious statistical constraint on any demographic comparison, and exactly the kind of limitation `docs/ETHICS.md` says we must state plainly rather than quietly exclude from a denominator. There's also real dirty data: `susp_age_group` has a stray value `"2026"` (obviously a data-entry/system error, 1 row) and `vic_age_group` has `"-960"`/`"-968"` sentinel-looking values (1 row each) — all need explicit filtering in the Milestone 2 pipeline, not silent inclusion. `vic_sex` also has non-binary codes beyond M/F (`E`, `D`, `L` — likely "unknown"/"business"/other administrative codes per NYPD's data dictionary, to be confirmed before using this field for anything).

- **Bottom line for the demographic-fields policy:** availability alone won't be the bottleneck — the bottleneck will be whether a large enough, sufficiently-known-demographic, confounder-controlled subset exists to say anything statistically meaningful. That's a Milestone 2 question once we're working with the full historical volume instead of a 20k-row window, and per `docs/ETHICS.md` we should be ready for the honest answer to be "not really, here's why."